In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.optimize import least_squares

sys.path.insert(0, str(Path("../../nogse_pipeline/src")))
from models.model_fitting import M_ogse_mixed_offset
from plotting.publication.tc_param_vars import _roi_color_map, DEFAULT_BRAIN_MARKERS

In [ ]:
MASTER_PATH  = Path("../../analysis/brains/ogse_experiments/master.long.parquet")
OUT_DIR      = Path("../../analysis/brains/ogse_experiments/mixed_joint_fits")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIRECTIONS  = ["tra", "long"]
N_LIST      = [4, 8]
D0_FIXED    = 3.2e-12   # m²/ms, fixed
RN_FIXED    = 15.0      # fixed Rician noise floor
ALPHA_FIXED = 0         # None → alpha is fitted; float → alpha is pinned
M0_FIXED    = None      # None → M0 is fitted; float → M0 is pinned (e.g. 1.0 to normalise)

In [ ]:
df = pd.read_parquet(MASTER_PATH)

data = df[
    (df.row_kind == "signal_rotated") &
    (df.direction.isin(DIRECTIONS)) &
    (df.N.isin(N_LIST)) &
    (df.stat == "avg")
].copy()

data["G_eff"] = data.g_thorsten * data.grad_correction_factor

print(f"Rows loaded: {len(data)}")
print("Groups (subj, roi, dir):", data.groupby(["subj", "roi", "direction"]).ngroups)
data[["subj", "roi", "direction", "td_ms", "N", "G_eff", "value"]]

In [ ]:
def _unpack(params):
    """Extract (tc, alpha, M0, C) from the optimizer parameter vector."""
    idx = 0
    tc = np.exp(params[idx]); idx += 1
    if ALPHA_FIXED is None:
        alpha = params[idx]; idx += 1
    else:
        alpha = ALPHA_FIXED
    if M0_FIXED is None:
        M0 = params[idx]; idx += 1
    else:
        M0 = M0_FIXED
    C = params[idx]
    return tc, alpha, M0, C


def fit_td(td, curves_for_td):
    """Fit M_ogse_mixed_offset for ONE (subj, roi, dir, td) — N=4 and N=8 concatenated."""

    def residuals(params):
        tc, alpha, M0, C = _unpack(params)
        res = []
        for c in curves_for_td:
            x = td / c["N"]
            with np.errstate(over="ignore", invalid="ignore"):
                y_hat = M_ogse_mixed_offset(
                    td, c["G"], c["N"], x, tc, alpha, M0, D0_FIXED, C, RN_FIXED
                )
            res.append(y_hat - c["y"])
        out = np.concatenate(res)
        return np.where(np.isfinite(out), out, 1e6)

    M0_init = max(c["y"].max() for c in curves_for_td)
    x0    = [np.log(1.0)];  lower = [np.log(0.5)];  upper = [np.log(500.0)]
    if ALPHA_FIXED is None:
        x0 += [0.5];     lower += [0.0];     upper += [1.0]
    if M0_FIXED is None:
        x0 += [M0_init]; lower += [0.0];     upper += [np.inf]
    x0 += [0.0];         lower += [-np.inf]; upper += [np.inf]

    return least_squares(residuals, x0, bounds=(lower, upper), method="trf", max_nfev=10000)

In [ ]:
rows      = []
fit_store = {}

for (subj, roi, direction), grp in data.groupby(["subj", "roi", "direction"]):
    tds = sorted(float(t) for t in grp.td_ms.unique())
    fit_store[(subj, roi, direction)] = {"tds": tds, "fits": {}}

    for td in tds:
        sub = grp[np.isclose(grp.td_ms.astype(float), td)]
        curves = []
        for N in N_LIST:
            s = sub[sub.N == N].sort_values("G_eff")
            if len(s):
                curves.append(dict(N=int(N), G=s.G_eff.values, y=s.value.values))
        if not curves:
            continue

        result = fit_td(td, curves)
        tc, alpha, M0, C = _unpack(result.x)
        print(
            f"{subj:6s}  {roi:25s}  {direction}  td={td:5.0f} ms  "
            f"tc={tc:.3f} ms  alpha={alpha:.3f}  cost={result.cost:.3e}"
        )

        fit_store[(subj, roi, direction)]["fits"][td] = dict(
            curves=curves, tc=tc, alpha=alpha, M0=M0, C=C
        )
        for c in curves:
            rows.append(dict(
                subj=subj, roi=roi, direction=direction,
                td_ms=td, N=c["N"],
                tc_ms=tc, alpha=alpha, D0_m2ms=D0_FIXED,
                M0=M0, C=C, RN=RN_FIXED,
                cost=float(result.cost), success=bool(result.success),
            ))

In [ ]:
results_df = pd.DataFrame(rows)
results_df.to_excel(OUT_DIR / "fit_results.xlsx", index=False)
print(f"Saved {len(results_df)} rows to {OUT_DIR / 'fit_results.xlsx'}")
results_df

In [ ]:
G_plot = np.linspace(0, float(data.G_eff.max()), 300)

for (subj, roi, direction), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    fig, axes = plt.subplots(1, n_tds, figsize=(4 * n_tds, 3.5), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, td in zip(axes, tds):
        fit = store["fits"].get(td)
        if fit is None:
            continue
        for N, color in zip(N_LIST, ["C0", "C1"]):
            c = next((x for x in fit["curves"] if x["N"] == N), None)
            if c is None:
                continue
            ax.scatter(c["G"], c["y"], color=color, s=20, zorder=3, label=f"N={N} data")
            y_hat = M_ogse_mixed_offset(
                td, G_plot, N, td / N, fit["tc"], fit["alpha"], fit["M0"], D0_FIXED, fit["C"], RN_FIXED
            )
            ax.plot(G_plot, y_hat, color=color, label=f"N={N} fit")
        ax.set_title(
            f"td = {td:.1f} ms\ntc = {fit['tc']:.2f} ms   α = {fit['alpha']:.3f}",
            fontsize=8
        )
        ax.set_xlabel("G_eff [mT/m]")

    axes[0].set_ylabel("value [a.u.]")
    axes[0].legend(fontsize=6)
    fig.suptitle(f"{subj}  |  {roi}  |  {direction}", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"fit_{subj}_{roi}_{direction}.png", dpi=120)
    plt.close(fig)

print("Fit plots saved.")

In [ ]:
rois     = sorted(results_df.roi.unique())
subjects = sorted(results_df.subj.unique())
roi_colors = _roi_color_map(rois)

for direction in DIRECTIONS:
    sub = results_df[results_df.direction == direction]
    fig, axes = plt.subplots(1, len(rois), figsize=(4 * len(rois), 4), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, roi in zip(axes, rois):
        base_color = roi_colors[roi]
        cmap = mcolors.LinearSegmentedColormap.from_list("", ["#cccccc", base_color])
        for i, subj in enumerate(subjects):
            shade = cmap(0.3 + 0.7 * i / max(1, len(subjects) - 1))
            g = (
                sub[(sub.roi == roi) & (sub.subj == subj)]
                .drop_duplicates("td_ms")
                .sort_values("td_ms")
            )
            if g.empty:
                continue
            marker = DEFAULT_BRAIN_MARKERS[i % len(DEFAULT_BRAIN_MARKERS)]
            ax.scatter(g.td_ms, g.tc_ms, color=shade, marker=marker, label=subj, s=60, zorder=3)
            ax.plot(g.td_ms, g.tc_ms, color=shade, linewidth=0.8)
        ax.set_title(roi, fontsize=9)
        ax.set_xlabel("td [ms]")

    axes[0].set_ylabel("tc [ms]")
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, fontsize=7, loc="upper right", ncol=1)
    fig.suptitle(f"tc vs td — direction: {direction}", fontsize=10)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"tc_vs_td_{direction}.png", dpi=120)
    plt.show()

In [ ]:
for (subj, roi, direction), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    fig, axes = plt.subplots(1, n_tds, figsize=(4 * n_tds, 3.5), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, td in zip(axes, tds):
        fit = store["fits"].get(td)
        if fit is None:
            continue
        M0 = fit["M0"]
        for N, color in zip(N_LIST, ["C0", "C1"]):
            c = next((x for x in fit["curves"] if x["N"] == N), None)
            if c is None:
                continue
            ax.scatter(c["G"], c["y"] / M0, color=color, s=20, zorder=3, label=f"N={N} data")
            y_hat = M_ogse_mixed_offset(
                td, G_plot, N, td / N,
                fit["tc"], fit["alpha"], 1.0, D0_FIXED, fit["C"] / M0, RN_FIXED / M0
            )
            ax.plot(G_plot, y_hat, color=color, label=f"N={N} fit (M0=1)")
        ax.set_title(f"td = {td:.1f} ms", fontsize=8)
        ax.set_xlabel("G_eff [mT/m]")

    axes[0].set_ylabel("value / M0  [a.u.]")
    axes[0].legend(fontsize=6)
    fig.suptitle(
        f"{subj}  |  {roi}  |  {direction}  |  normalised (M0 = 1)",
        fontsize=9
    )
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"fit_norm_{subj}_{roi}_{direction}.png", dpi=120)
    plt.close(fig)

print("Normalised fit plots saved.")